##Configuración del Entorno Espacial (Google Earth Engine)

###1. Importación y Autenticación

In [9]:
# La librería 'ee' (Earth Engine) en colab ya viene preinstalada
import ee


In [10]:
# 1. Ejecutar la autenticación
ee.Authenticate()

###2. Inicialización del Entorno



In [11]:
# Reemplaza el valor de proyect con el ID del proyecto
try:
    ee.Initialize(project='decisive-force-313716')
    print("¡Conexión a los satélites de Google establecida con éxito!")
except Exception as e:
    print("Error al inicializar. Revisa la autenticación o el ID del proyecto.")
    print(e)

¡Conexión a los satélites de Google establecida con éxito!


3. Prueba de Conexión (Altitud de Jauja => entre 3300 y 3400 msnm)

In [12]:
# 1. Definir un punto geográfico (Longitud, Latitud) para Jauja
jauja_point = ee.Geometry.Point([-75.500, -11.776])

# 2. Llamar al dataset satelital de elevación de la NASA (SRTM)
dataset_elevacion = ee.Image('CGIAR/SRTM90_V4')

# 3. Extraer el valor exacto de la elevación en ese punto
altitud = dataset_elevacion.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=jauja_point,
    scale=90 # Resolución de 90 metros
).getInfo()

print(f"Altitud extraída satelitalmente para Jauja: {altitud.get('elevation')} metros sobre el nivel del mar.")

Altitud extraída satelitalmente para Jauja: 3390 metros sobre el nivel del mar.


##Extracción de Variables Climáticas (Satélite)

###1. Definir la Región y el Dataset

In [13]:
import pandas as pd

# 1. Definir un área de interés (Buffer de 10 km alrededor de Jauja)
punto_jauja = ee.Geometry.Point([-75.500, -11.776])
region_estudio = punto_jauja.buffer(10000) # 10,000 metros (10 km)

# 2. Llamar al dataset TerraClimate (Clima mensual global)
# Seleccionamos: 'pr' (Precipitación), 'tmmx' (Temp. Max), 'tmmn' (Temp. Min)
dataset_clima = (ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE')
                 .filterBounds(region_estudio)
                 .filterDate('2021-01-01', '2026-09-30')
                 .select(['pr', 'tmmx', 'tmmn']))

print(f"Imágenes mensuales encontradas: {dataset_clima.size().getInfo()}")


Imágenes mensuales encontradas: 48


###2. Reducción Espacial (Procesamiento en la nube de Google)

In [14]:
# Función para calcular el promedio de cada mes en nuestra región
def extraer_promedios(imagen):
    # Calcular el promedio de los píxeles dentro de la región
    promedios = imagen.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region_estudio,
        scale=4000, # TerraClimate tiene una resolución de ~4km
        maxPixels=1e9
    )

    # TerraClimate guarda las temperaturas multiplicadas por 10 (ej. 250 = 25.0°C)
    # Debemos escalarlas correctamente
    tmax = ee.Number(promedios.get('tmmx')).multiply(0.1)
    tmin = ee.Number(promedios.get('tmmn')).multiply(0.1)
    precip = ee.Number(promedios.get('pr'))

    # Extraer la fecha de la imagen
    fecha = imagen.date().format('YYYY-MM')

    # Retornar un diccionario con los datos limpios
    return ee.Feature(None, {
        'fecha': fecha,
        'precipitacion_mm': precip,
        'temp_max_c': tmax,
        'temp_min_c': tmin
    })

# Aplicar la función a toda la colección de imágenes
datos_procesados = dataset_clima.map(extraer_promedios)


###3. Conversión a Pandas DataFrame


In [15]:
# Descargar los datos procesados como una lista
lista_datos = datos_procesados.getInfo()['features']

# Extraer solo las propiedades (nuestros valores) de cada elemento
filas = [item['properties'] for item in lista_datos]

# Convertir a Pandas DataFrame
df_clima = pd.DataFrame(filas)

# Ordenar por fecha por si acaso
df_clima = df_clima.sort_values(by='fecha').reset_index(drop=True)

# ¡Listo! Aquí está tu tabla climática
print(df_clima.head())
df_clima.to_csv('datos_clima.csv', index=False)

     fecha  precipitacion_mm  temp_max_c  temp_min_c
0  2021-01         97.319107   16.642422    4.620213
1  2021-02        106.955740   16.554772    5.412389
2  2021-03        148.084371   16.087315    4.380083
3  2021-04         52.350524   16.994527    3.380300
4  2021-05         12.865442   16.780202    1.569018
